# 63 · RAG over SEC 10-K filings (B113 Phase 3 — finance domain)

The finance domain's **filings-RAG** lane. `datasets_finance_edgar_text_land` fetches each mega-cap's latest
**10-K**, strips the HTML, and chunks the narrative **section-aware** (`edgar_text_parse` — Business / Risk
Factors / Legal Proceedings / MD&A / Market Risk); the `datasets_lib` vector fan-out embeds each chunk with
**bge-small (384-dim)** and upserts it to **Qdrant** (collection `datasets_finance_filings_text`), Weaviate, and
LanceDB. Here we retrieve over that corpus and answer **with citations** via the LiteLLM `wl-rag` lane.

Structured financials live in the Phase-2 `mart_company_financials`; this is the *narrative* half — the text a
human would actually read in the filing. See notebook **60** for the general RAG pattern this mirrors.

In [ ]:
%pip install -q sentence-transformers qdrant-client openai polars

## Connect — Qdrant corpus + the bge-small query embedder + the `wl-rag` gateway

The embedder MUST be the exact model the corpus was built with (**bge-small**, 384-dim) — query and index
geometry have to match. Note this differs from notebook 60's bge-base(768) corpus.

In [ ]:
import os
from qdrant_client import QdrantClient
from qdrant_client import models as qm
from openai import OpenAI
import polars as pl

# committed defaults = in-cluster DNS; a validation run overrides via env, never in the notebook
QDRANT_URL       = os.environ.get('QDRANT_URL', 'http://qdrant.weyland.svc.cluster.local:6333')
LITELLM_BASE_URL = os.environ.get('LITELLM_BASE_URL', 'http://litellm.weyland.svc.cluster.local:4000/v1')
LITELLM_KEY      = os.environ['LITELLM_MASTER_KEY']   # injected from litellm-creds in-pod

COLLECTION = 'datasets_finance_filings_text'   # the vector fan-out names it datasets_<domain>_<dataset>
EMBED_MODEL = 'BAAI/bge-small-en-v1.5'         # 384-dim — the EXACT model the loader embedded with
GEN_MODEL   = 'wl-rag'                          # LiteLLM's retrieval-grounded generation lane

qdrant = QdrantClient(url=QDRANT_URL)
llm    = OpenAI(base_url=LITELLM_BASE_URL, api_key=LITELLM_KEY)

def snippet(t, n=140):
    return None if t is None else ' '.join(str(t).split())[:n]

## Confirm the corpus — the collection, its size, and the sections it carries

In [ ]:
info = qdrant.get_collection(COLLECTION)
count = qdrant.count(COLLECTION).count
print(f'collection {COLLECTION!r}: {count:,} chunks, dim {info.config.params.vectors.size}')

# peek at a few payloads to see the citation fields the loader carried (ticker / section / accn / text)
sample = qdrant.scroll(COLLECTION, limit=5, with_payload=True)[0]
pl.DataFrame([
    {'ticker': p.payload.get('ticker'), 'section': p.payload.get('section'),
     'accn': p.payload.get('accn'), 'text': snippet(p.payload.get('text'))}
    for p in sample])

## Retrieve — a natural-language question over the filings

Bare text + `normalize_embeddings=True` — exactly how the loader embedded the chunks (cosine space).

In [ ]:
from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer(EMBED_MODEL)

question = 'What supply chain, manufacturing, and component-sourcing risks do these companies disclose?'
query_vec = embedder.encode(question, normalize_embeddings=True).tolist()

TOP_K = 6
hits = qdrant.query_points(COLLECTION, query=query_vec, limit=TOP_K, with_payload=True).points
pl.DataFrame([
    {'rank': i, 'score': round(h.score, 4), 'ticker': h.payload.get('ticker'),
     'section': h.payload.get('section'), 'snippet': snippet(h.payload.get('text'))}
    for i, h in enumerate(hits, 1)])

## Answer — grounded generation with citations

Each retrieved chunk becomes a numbered, **cited** context passage (ticker · section). The model is told to
answer only from the passages and cite the `[n]` markers it uses.

In [ ]:
context = '\n\n'.join(
    f"[{i}] {h.payload.get('ticker')} 10-K \u00b7 {h.payload.get('section')}\n{h.payload.get('text')}"
    for i, h in enumerate(hits, 1))

SYSTEM = ('You are a precise financial-filings assistant. Answer the question using ONLY the numbered context '
          'passages, which are excerpts from SEC 10-K filings. Cite the passages you use by their [n] marker '
          'and name the company. If the context does not contain the answer, say so plainly instead of guessing.')
USER = f'Context:\n{context}\n\nQuestion: {question}'

resp = llm.chat.completions.create(model=GEN_MODEL,
    messages=[{'role':'system','content':SYSTEM},{'role':'user','content':USER}], temperature=0.1)
print('GROUNDED answer (wl-rag, retrieved 10-K passages in the prompt):\n')
print((resp.choices[0].message.content or '').strip())

## Contrast — the same question with no retrieved context

The model's parametric guess, ungrounded — no filing text, no citations. The gap is the value of retrieval.

In [ ]:
resp_raw = llm.chat.completions.create(model=GEN_MODEL,
    messages=[{'role':'user','content':question}], temperature=0.1)
print('NO-CONTEXT answer (wl-rag, question only):\n')
print((resp_raw.choices[0].message.content or '').strip())

## Section-aware retrieval pays off — filter to just **Risk Factors**

Because the chunker tagged every chunk with its 10-K Item, the same query can be scoped to one section — e.g.
retrieve *only* Risk Factors disclosures, excluding MD&A/Business boilerplate.

In [ ]:
flt = qm.Filter(must=[qm.FieldCondition(key='section', match=qm.MatchValue(value='Risk Factors'))])
rf = qdrant.query_points(COLLECTION, query=query_vec, limit=5, with_payload=True, query_filter=flt).points
print('Risk-Factors-only hits:')
pl.DataFrame([
    {'ticker': h.payload.get('ticker'), 'section': h.payload.get('section'),
     'score': round(h.score,4), 'snippet': snippet(h.payload.get('text'))} for h in rf])